<a href="https://colab.research.google.com/github/alaycheme25/chemeng277_batteryproject/blob/main/machine_learning_3_5_after_meeting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install mp-api pymatgen periodictable paretoset

### Importing necessary packages and files

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.decomposition import PCA
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import paretoset
import periodictable
from pymatgen.core import Structure
import ast
from sklearn.linear_model import ElasticNet

In [3]:
pdata = pd.read_excel("MATSCI_176_Project_Data.xlsx")
pdata.head()

,battery_formula,working_ion,elements,nelements,formula_charge,formula_discharge,max_delta_volume,average_voltage,capacity_grav,capacity_vol,...,stability_charge,stability_discharge,electrode_density,material_density,working_ion_cost_per_kWh,structure,formation_energy_per_atom,band_gap,energy_density,battery_cost_per_L
0,Li0-3Ag,Li,['Ag'],1.0,Ag,Li3Ag,2.736089,0.210565,624.785871,1984.816229,...,0.003609,0.000000,3.176794,2.592432,10.0,"{'@module': 'pymatgen.core.structure', '@class...",0.333634,0,417.933696,4.179337
1,Li0-3Sb,Li,['Sb'],1.0,Sb,Li3Sb,1.569237,1.015953,563.913254,1890.293779,...,0.328351,0.000000,3.352100,3.378028,10.0,"{'@module': 'pymatgen.core.structure', '@class...",-0.744135,0.7074,1920.448911,19.204489
2,Li0-1Bi,Li,['Bi'],1.0,Bi,LiBi,0.368255,0.796796,124.126099,923.538664,...,0.049545,0.000000,7.440326,2.237712,10.0,"{'@module': 'pymatgen.core.structure', '@class...",-0.034716,0,735.871648,7.358716
3,Li0-3Ce,Li,['Ce'],1.0,Ce,Li3Ce,2.951183,-0.444846,499.595769,1295.168213,...,0.000000,0.333634,2.592432,2.583328,10.0,"{'@module': 'pymatgen.core.structure', '@class...",-0.330999,0,576.149771,5.761498
4,Li0-3Ca,Li,['Ca'],1.0,Ca,Li3Ca,1.513956,-0.064931,1320.248328,1271.741142,...,0.000000,0.057445,0.963259,7.527518,10.0,"{'@module': 'pymatgen.core.structure', '@class...",-0.415156,0,82.574983,0.825750


In [4]:
structures = pdata["structure"]
print(structures.shape)
structure_dict_test = ast.literal_eval(structures[1])
structure_test = Structure.from_dict(structure_dict_test)
print(structure_test)
print(structure_test.lattice.abc)
print(structure_test.volume)

(5786,)
Full Formula (Li3 Sb1)
Reduced Formula: Li3Sb
abc   :   4.627959   4.627959   4.627959
angles:  60.000007  60.000010  60.000007
pbc   :       True       True       True
Sites (4)
  #  SP       a      b     c    magmom
---  ----  ----  -----  ----  --------
  0  Li    0.5    0.5   0.5          0
  1  Li    0.25   0.25  0.25        -0
  2  Li    0.75   0.75  0.75        -0
  3  Sb    0     -0     0            0
(4.627959209506349, 4.627958733729667, 4.62795882)
70.08959778250679


In [5]:
def get_volume(structure_str):
    try:
        structure_dict = ast.literal_eval(structure_str)
        structure = Structure.from_dict(structure_dict)
        return structure.volume
    except:
        return np.nan
structure_volums = pdata["structure"].apply(get_volume) # Volumes in Angstrom^3
def get_abc(structure_str):
    try:
        structure_dict = ast.literal_eval(structure_str)
        structure = Structure.from_dict(structure_dict)
        return structure.lattice.abc
    except:
        return np.nan
# structure_abc = pdata["structure"].apply(get_abc) # abc values in Angstrom
# structure_abc_df = pd.DataFrame(structure_abc.tolist(), columns=['a', 'b', 'c'])
pdata = pdata.drop(columns=["structure"], axis=1)

In [6]:
structure_volums = structure_volums.to_frame()
structure_volums = structure_volums.fillna(0)
structure_volums = structure_volums["structure"]
pdata["structure_volume"] = structure_volums.values

In [7]:
pdata.head()

,battery_formula,working_ion,elements,nelements,formula_charge,formula_discharge,max_delta_volume,average_voltage,capacity_grav,capacity_vol,...,stability_charge,stability_discharge,electrode_density,material_density,working_ion_cost_per_kWh,formation_energy_per_atom,band_gap,energy_density,battery_cost_per_L,structure_volume
0,Li0-3Ag,Li,['Ag'],1.0,Ag,Li3Ag,2.736089,0.210565,624.785871,1984.816229,...,0.003609,0.000000,3.176794,2.592432,10.0,0.333634,0,417.933696,4.179337,103.086779
1,Li0-3Sb,Li,['Sb'],1.0,Sb,Li3Sb,1.569237,1.015953,563.913254,1890.293779,...,0.328351,0.000000,3.352100,3.378028,10.0,-0.744135,0.7074,1920.448911,19.204489,70.089598
2,Li0-1Bi,Li,['Bi'],1.0,Bi,LiBi,0.368255,0.796796,124.126099,923.538664,...,0.049545,0.000000,7.440326,2.237712,10.0,-0.034716,0,735.871648,7.358716,58.627385
3,Li0-3Ce,Li,['Ce'],1.0,Ce,Li3Ce,2.951183,-0.444846,499.595769,1295.168213,...,0.000000,0.333634,2.592432,2.583328,10.0,-0.330999,0,576.149771,5.761498,1918.735425
4,Li0-3Ca,Li,['Ca'],1.0,Ca,Li3Ca,1.513956,-0.064931,1320.248328,1271.741142,...,0.000000,0.057445,0.963259,7.527518,10.0,-0.415156,0,82.574983,0.825750,47.631359


In [8]:
numeric_cols = [
    "material_density",
    "formation_energy_per_atom",
    "band_gap"
]

for col in numeric_cols:
    pdata[col] = pd.to_numeric(pdata[col], errors="coerce")
pdata = pdata.dropna()
pdata = pdata.reset_index(drop=True)
pdata.head()

,battery_formula,working_ion,elements,nelements,formula_charge,formula_discharge,max_delta_volume,average_voltage,capacity_grav,capacity_vol,...,stability_charge,stability_discharge,electrode_density,material_density,working_ion_cost_per_kWh,formation_energy_per_atom,band_gap,energy_density,battery_cost_per_L,structure_volume
0,Li0-3Ag,Li,['Ag'],1.0,Ag,Li3Ag,2.736089,0.210565,624.785871,1984.816229,...,0.003609,0.000000,3.176794,2.592432,10.0,0.333634,0.0000,417.933696,4.179337,103.086779
1,Li0-3Sb,Li,['Sb'],1.0,Sb,Li3Sb,1.569237,1.015953,563.913254,1890.293779,...,0.328351,0.000000,3.352100,3.378028,10.0,-0.744135,0.7074,1920.448911,19.204489,70.089598
2,Li0-1Bi,Li,['Bi'],1.0,Bi,LiBi,0.368255,0.796796,124.126099,923.538664,...,0.049545,0.000000,7.440326,2.237712,10.0,-0.034716,0.0000,735.871648,7.358716,58.627385
3,Li0-3Ce,Li,['Ce'],1.0,Ce,Li3Ce,2.951183,-0.444846,499.595769,1295.168213,...,0.000000,0.333634,2.592432,2.583328,10.0,-0.330999,0.0000,576.149771,5.761498,1918.735425
4,Li0-3Ca,Li,['Ca'],1.0,Ca,Li3Ca,1.513956,-0.064931,1320.248328,1271.741142,...,0.000000,0.057445,0.963259,7.527518,10.0,-0.415156,0.0000,82.574983,0.825750,47.631359


In [9]:
pdata_norm = pdata.iloc[:, 6:]
print(pdata_norm.columns)

Index(['max_delta_volume', 'average_voltage', 'capacity_grav', 'capacity_vol',
       'energy_grav', 'energy_vol', 'stability_charge', 'stability_discharge',
       'electrode_density', 'material_density', 'working_ion_cost_per_kWh',
       'formation_energy_per_atom', 'band_gap', 'energy_density',
       'battery_cost_per_L', 'structure_volume'],
      dtype='object')


#### Defining our scoring function(s) to analyze and their input datasets.
We don't want to train our model on the features which are used for scoring. Therefore, we need to define a new training dataset excluding these features. Our scoring model uses gravimetric energy, gravimetric capacity, battery cost per liter, energy density, and battery voltage, so we removed these from our training data set. Additionally, features like energy volume, capacity volume, and electrode density are correlated with battery cost and energy density (as they're used in the calculations for those features), so we removed those columns from our training data to avoid any overlap.

In [10]:
# Getting the energy grav, capacity grav, battery cost per L, energy density, and average voltage columns for scoring and removing those and additional columns from the training data
# energy_grav = pdata_norm["energy_grav"]
cap_grav = pdata_norm["capacity_grav"]
battery_cost = pdata_norm["battery_cost_per_L"]
energy_density = pdata_norm["energy_density"]
voltage = pdata_norm['average_voltage']
training_data = pdata_norm.drop(columns=["energy_grav", "capacity_grav", "battery_cost_per_L", "energy_density", "average_voltage", "energy_vol", "capacity_vol", "electrode_density", "max_delta_volume", "stability_charge"])

# Creating a new training dataset without cost to benchmark our scoring with and without cost, since our cost is not from the same source (Materials Project)
training_data_nocost = training_data.drop(columns=["working_ion_cost_per_kWh"])

In [11]:
def scoring_nocost(c_g,e_d,v):
    # e_g = e_g.to_numpy().astype(float)
    c_g = c_g.to_numpy().astype(float)
    e_d = e_d.to_numpy().astype(float)
    v = v.to_numpy().astype(float)
    score = 0.4*(np.abs(e_d)-np.min(e_d))/(np.max(e_d)-np.min(e_d)) + 0.4*(np.abs(c_g)-np.min(c_g))/((np.max(c_g))-np.min(c_g)) + 0.2*(np.abs(v)-np.min(np.abs(v)))/(np.max(v)-np.min(np.abs(v)))
    # score = abs(e_g) + abs(c_g) + abs(e_d) + abs(v)
    return pd.Series(score)

def scoring_withcost(c_g,b_c,e_d,v):
    # e_g = e_g.to_numpy().astype(float)
    c_g = c_g.to_numpy().astype(float)
    b_c = b_c.to_numpy().astype(float)
    e_d = e_d.to_numpy().astype(float)
    v = v.to_numpy().astype(float)
    score = np.abs(e_d)/np.max(e_d) + np.abs(c_g/b_c)/(np.max(c_g)/np.max(b_c)) + np.abs(v)/np.max(v)
    # score = abs(e_g) + abs(c_g) - abs(b_c) + abs(e_d) + abs(v)
    return pd.Series(score)

### Benchmarking the Ridge & Elastic Net Regressional Models
We chose ridge regression and elasticnet regression to both identify important features (from the ridge component) while zeroing out features which don't affect battery performance (features that don't affect our score.)

In [12]:
# Testing the no-cst scoring function and defining our fit & training datasets.
battery_score = scoring_nocost(cap_grav, energy_density, voltage)

fit_train, fit_test, score_train, score_test = train_test_split(
    training_data_nocost,
    battery_score,
    test_size=0.1,
    random_state=42
)

# Normalization of the fit and training datasets for better performance of the elastic net model, since it is sensitive to the scale of the features
norm_fit_train = pd.DataFrame(
    StandardScaler().fit_transform(fit_train),
    columns=fit_train.columns
)
norm_fit_test = pd.DataFrame(
    StandardScaler().fit_transform(fit_test),
    columns=fit_test.columns
)

In [13]:
ridge = linear_model.Ridge(alpha=1.0).fit(norm_fit_train, score_train, sample_weight=None)
score_pred = ridge.score(norm_fit_test, score_test)
print(f"Effective R^2 score: {score_pred:.3f}")

Effective R^2 score: 0.028


In [14]:
best = np.where(battery_score == np.max(battery_score))[0][0]
# print(pdata_norm.iloc[best])
print(pdata.iloc[best])
# print(len(pdata["capacity_grav"]))
# print(len(pdata_norm["capacity_grav"]))
print("Battery Score:", np.max(battery_score))
# print(energy_grav.iloc[best], cap_grav.iloc[best], energy_density.iloc[best], voltage.iloc[best])

battery_formula                Li1-3Ti2(PO4)3
working_ion                                Li
elements                     ['Ti', 'P', 'O']
nelements                                 3.0
formula_charge                    LiTi2(PO4)3
formula_discharge                Li3Ti2(PO4)3
max_delta_volume                     0.086321
average_voltage                     33.065771
capacity_grav                      133.516371
capacity_vol                       421.348595
energy_grav                       4414.821755
energy_vol                       13932.216223
stability_charge                     3.516172
stability_discharge                  0.074915
electrode_density                    3.155782
material_density                     5.482777
working_ion_cost_per_kWh                 10.0
formation_energy_per_atom           -2.067682
band_gap                                  0.0
energy_density                   13932.216223
battery_cost_per_L                 139.322162
structure_volume                  

Looks pretty decent! We expected a lithium battery to be the best performing battery due to it's electrochemical properties.

## Performing K-folds cross validation on our ridge regressional model.
Choosing k-folds is important to determine if our train-validation split was truely indicative of the full system.

In [15]:
X = training_data_nocost
y = battery_score
y = y.fillna(0)
y = y.astype(float)

In [16]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", linear_model.Ridge(alpha=3.0)) # We chose alpha = 3.0 to weight against sparsity in our features.
])

In [17]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []
MAE = []
MSE = []
bestmaterial = []

for train_idx, test_idx in kf.split(X):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    #Find best material in predicted set
    bestmaterial.append(X_test.iloc[np.where(y_pred == np.max(y_pred))[0][0]])

    r2_scores.append(r2_score(y_test, y_pred))
    MSE.append(mean_squared_error(y_test, y_pred))
    MAE.append(mean_absolute_error(y_test, y_pred))

print("Mean R2:", np.mean(r2_scores))
print("Mean MSE:", np.mean(MSE))
print("Mean MAE:", np.mean(MAE))
bestmaterial_df = pd.DataFrame(bestmaterial)
bestmaterial_df

Mean R2: 0.03525312350408598
Mean MSE: 0.002746618148578866
Mean MAE: 0.03890975954867328


,stability_discharge,material_density,formation_energy_per_atom,band_gap,structure_volume
4098,0.00000,4.322285,3.615826,0.0,101.329084
604,0.00000,1.885992,0.700918,0.0,109.893911
6,0.01007,1.481680,0.426453,0.0,36.797188
1796,0.01340,3.172677,0.838711,0.0,176.809733
608,0.00000,1.816624,1.064513,0.0,74.022197


In [18]:
# Converting the best material features back to the original scale for comparison with the original dataset, since we normalized the training & test data.
# To do so, we took the best material from each fold and compared their features to the original datasets to find the approximated battery  (the battery formula with the smallest minimum difference between predicted best and the rest of the dataset).
for i in range(len(bestmaterial)):
    best_material = bestmaterial_df.iloc[i]
    min_diff = float('inf')
    best_match_index = -1
    for j in range(len(pdata)):
        diff = np.sum(np.abs(best_material - pdata.iloc[j, 6:6+len(best_material)]))
        if diff < min_diff:
            min_diff = diff
            best_match_index = j
    print(f"Best material in fold {i+1}:")
    print(pdata.iloc[best_match_index, 0])

Best material in fold 1:
Li0-3Ag
Best material in fold 2:
Li0-3Ag
Best material in fold 3:
Li0-3Ag
Best material in fold 4:
Li0-3Ag
Best material in fold 5:
Li0-3Ag


## Testing Elastic Net Regression in the same way as Ridge Regression

In [19]:
alpha = 1
l1_ratio = 0.5  # 0=ridge-like, 1=lasso-like

enet_pipeline = Pipeline([
    ("enet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000, random_state=42))
])

score_pred_enet = enet_pipeline.fit(fit_train, score_train).score(fit_test, score_test)
print("ElasticNet test R^2:", score_pred_enet)

ElasticNet test R^2: 0.0037803172393754414


In [20]:
#replace Ridge KFold with Elastic Net KFold

X = training_data_nocost
y = battery_score.fillna(0).astype(float)

# Elastic Net hyperparameters
alpha = 1
l1_ratio = 0.5

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("enet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000, random_state=42))
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []
MAE = []
MSE = []
bestmaterial = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    # store best predicted sample in this fold (same idea as your ridge code)
    bestmaterial.append(X_test.iloc[np.argmax(y_pred)])

    r2_scores.append(r2_score(y_test, y_pred))
    MSE.append(mean_squared_error(y_test, y_pred))
    MAE.append(mean_absolute_error(y_test, y_pred))

print("Mean R^2:", np.mean(r2_scores))
print("Mean MSE:", np.mean(MSE))
print("Mean MAE:", np.mean(MAE))

bestmaterial_df = pd.DataFrame(bestmaterial)
bestmaterial_df

Mean R^2: -0.0027740444405376153
Mean MSE: 0.002853115912168843
Mean MAE: 0.03953833940083366


,stability_discharge,material_density,formation_energy_per_atom,band_gap,structure_volume
8,0.002960,0.946023,0.046672,0.0000,106.898527
19,0.020862,4.829614,-1.304004,0.0000,31.265978
0,0.000000,2.592432,0.333634,0.0000,103.086779
1,0.000000,3.378028,-0.744135,0.7074,70.089598
3,0.333634,2.583328,-0.330999,0.0000,1918.735425


In [21]:
for i in range(len(bestmaterial)):
    best_material = bestmaterial_df.iloc[i]
    min_diff = float('inf')
    best_match_index = -1
    for j in range(len(pdata)):
        diff = np.sum(np.abs(best_material - pdata.iloc[j, 6:6+len(best_material)]))
        if diff < min_diff:
            min_diff = diff
            best_match_index = j
    print(f"Best material in fold {i+1}:")
    print(pdata.iloc[best_match_index, 0])

Best material in fold 1:
Li0-3Ag
Best material in fold 2:
Li0-3Ag
Best material in fold 3:
Li0-3Ag
Best material in fold 4:
Li0-3Ag
Best material in fold 5:
Li0-3Ag


## Testing on other DFT-based features from materials project:
Testing to see if our model & scoring function can predict the quality of a battery based on untrained features, which still correlate with battery lifetime and hence the score.

In [22]:
test_data = pd.read_csv("test_data.csv")
test_data.head()

,formula_pretty,density,material_id,structure,formation_energy_per_atom,energy_above_hull,band_gap,fields_not_requested
0,K3LiP2O7,2.511335,mp-cgaow,"{'@module': 'pymatgen.core.structure', '@class...",-2.671974,0.0,5.0761,"['builder_meta', 'nsites', 'elements', 'neleme..."
1,Rb2LiSbCl6,3.060150,mp-clfam,"{'@module': 'pymatgen.core.structure', '@class...",-1.878165,0.0,2.9411,"['builder_meta', 'nsites', 'elements', 'neleme..."
2,Rb2LiBiCl6,3.317268,mp-clfaq,"{'@module': 'pymatgen.core.structure', '@class...",-1.932272,0.0,3.6154,"['builder_meta', 'nsites', 'elements', 'neleme..."
3,K2LiBiCl6,2.867108,mp-clfht,"{'@module': 'pymatgen.core.structure', '@class...",-1.894874,0.0,3.6182,"['builder_meta', 'nsites', 'elements', 'neleme..."
4,K2LiGaF6,3.349747,mp-clfrc,"{'@module': 'pymatgen.core.structure', '@class...",-3.104615,0.0,6.1271,"['builder_meta', 'nsites', 'elements', 'neleme..."


In [23]:
structures = test_data["structure"]
print(structures.shape)
structure_dict_test = ast.literal_eval(structures[1])
structure_test = Structure.from_dict(structure_dict_test)
print(structure_test)
print(structure_test.lattice.abc)
print(structure_test.volume)

(556,)
Full Formula (Rb2 Li1 Sb1 Cl6)
Reduced Formula: Rb2LiSbCl6
abc   :   7.325951   7.325951   7.325951
angles:  60.000000  60.000000  60.000000
pbc   :       True       True       True
Sites (10)
  #  SP           a          b         c    magmom
---  ----  --------  ---------  --------  --------
  0  Rb    0.75       0.75      0.75             0
  1  Rb    0.25       0.25      0.25             0
  2  Li    0.5        0.5       0.5             -0
  3  Sb    0         -0         0               -0
  4  Cl    0.745526   0.254474  0.254474        -0
  5  Cl    0.254474   0.254474  0.745526        -0
  6  Cl    0.254474   0.745526  0.745526        -0
  7  Cl    0.254474   0.745526  0.254474        -0
  8  Cl    0.745526   0.254474  0.745526        -0
  9  Cl    0.745526   0.745526  0.254474        -0
(7.3259512676535365, 7.3259512676535365, 7.3259512676535365)
278.0206653745919


In [24]:
def get_volume(structure_str):
    try:
        structure_dict = ast.literal_eval(structure_str)
        structure = Structure.from_dict(structure_dict)
        return structure.volume
    except:
        return np.nan
structure_volums = test_data["structure"].apply(get_volume) # Volumes in Angstrom^3
def get_abc(structure_str):
    try:
        structure_dict = ast.literal_eval(structure_str)
        structure = Structure.from_dict(structure_dict)
        return structure.lattice.abc
    except:
        return np.nan

test_data = test_data.drop(columns=["structure"], axis=1)

In [25]:
structure_volums = structure_volums.to_frame()
structure_volums = structure_volums.fillna(0)
structure_volums = structure_volums["structure"]
test_data["structure_volume"] = structure_volums.values

In [26]:
test_data.head()

,formula_pretty,density,material_id,formation_energy_per_atom,energy_above_hull,band_gap,fields_not_requested,structure_volume
0,K3LiP2O7,2.511335,mp-cgaow,-2.671974,0.0,5.0761,"['builder_meta', 'nsites', 'elements', 'neleme...",394.322773
1,Rb2LiSbCl6,3.060150,mp-clfam,-1.878165,0.0,2.9411,"['builder_meta', 'nsites', 'elements', 'neleme...",278.020665
2,Rb2LiBiCl6,3.317268,mp-clfaq,-1.932272,0.0,3.6154,"['builder_meta', 'nsites', 'elements', 'neleme...",300.131828
3,K2LiBiCl6,2.867108,mp-clfht,-1.894874,0.0,3.6182,"['builder_meta', 'nsites', 'elements', 'neleme...",293.543482
4,K2LiGaF6,3.349747,mp-clfrc,-3.104615,0.0,6.1271,"['builder_meta', 'nsites', 'elements', 'neleme...",133.275008


In [27]:
stability_discharge = test_data["energy_above_hull"]
material_density = test_data["density"]
formation_energy_per_atom = test_data["formation_energy_per_atom"]
band_gap = test_data["band_gap"]
structure_volume = test_data["structure_volume"]
test_data_short = pd.DataFrame({
    "stability_discharge": stability_discharge,
    "material_density": material_density,
    "formation_energy_per_atom": formation_energy_per_atom,
    "band_gap": band_gap,
    "structure_volume": structure_volume,
})

In [28]:
test_data_short.head()

,stability_discharge,material_density,formation_energy_per_atom,band_gap,structure_volume
0,0.0,2.511335,-2.671974,5.0761,394.322773
1,0.0,3.060150,-1.878165,2.9411,278.020665
2,0.0,3.317268,-1.932272,3.6154,300.131828
3,0.0,2.867108,-1.894874,3.6182,293.543482
4,0.0,3.349747,-3.104615,6.1271,133.275008


In [29]:
test_pred = ridge.predict(test_data_short)

In [30]:
print(test_pred)

[-1.84735006e+00 -1.28204546e+00 -1.38986404e+00 -1.35641509e+00
 -5.93353689e-01 -1.25471295e+00 -1.37865852e+00 -6.64588755e-01
 -1.73233951e+00 -1.71917878e+00 -1.41244178e+00 -2.16323438e+00
 -1.40203481e+00 -1.43368618e+00 -1.31015657e+00 -2.18046631e+00
 -1.39185721e+00 -6.97473288e-01 -1.30628558e+00 -7.44827987e-01
 -7.11607866e-01 -1.58702447e+00 -1.36017869e+00 -1.58950866e+00
 -1.23863388e+00 -7.15938889e-01 -6.64981216e-01 -1.26609236e+00
 -8.10971597e-01 -9.52979043e-01 -1.35980891e+00 -2.40377233e+00
 -6.92922109e-01 -3.48242397e+00 -1.58521711e+00 -3.55065403e+00
 -1.47836423e+00 -1.31704076e+00 -1.01782104e+00 -3.42219611e+00
 -3.53256586e-01 -3.58120115e-01 -3.41410670e-01 -3.88824271e-01
 -3.84903476e-01 -3.37445511e-01 -3.29417371e-01 -3.97370648e-01
 -3.92905050e-01 -3.33831683e-01 -4.03759262e-01 -9.97658812e-02
 -1.03884264e-01 -3.82738422e-01 -5.77358395e-01 -9.75155386e+00
 -4.24571016e+00 -9.34210034e+00 -2.33064600e+00 -2.63406703e+00
 -1.81320623e-01 -2.82993

In [31]:
norm_test_data = pd.DataFrame(
    StandardScaler().fit_transform(test_data_short),
    columns=test_data_short.columns
)

In [32]:
test_pred = ridge.predict(norm_test_data)

In [33]:
test_pred

array([ 0.08119328,  0.08746653,  0.08626836,  0.08765432,  0.07846149,
        0.08350445,  0.08110218,  0.0785587 ,  0.08234014,  0.0822375 ,
        0.08123731,  0.08463925,  0.08129874,  0.08158711,  0.0816556 ,
        0.08471387,  0.0812196 ,  0.0792472 ,  0.08224482,  0.07726189,
        0.07677811,  0.08286787,  0.08096173,  0.08555299,  0.08658456,
        0.07447835,  0.07420984,  0.08666636,  0.0940181 ,  0.09166927,
        0.07426681,  0.07896233,  0.08751396,  0.07211257,  0.08677632,
        0.07712197,  0.08253023,  0.08413103,  0.09873133,  0.08414129,
        0.09121542,  0.08510445,  0.08401496,  0.08188358,  0.08170585,
        0.08369489,  0.08315415,  0.08268364,  0.08208583,  0.08345101,
        0.08290747,  0.08648949,  0.07933286,  0.09467553,  0.0947228 ,
        0.07995993,  0.09779667,  0.08846267,  0.07215369,  0.07499522,
        0.09916834,  0.09875268,  0.09764405,  0.09996516,  0.10137963,
        0.07113697,  0.0982829 ,  0.09983699,  0.09225291,  0.09